# FastAPI Tutorial - Step by Step

This notebook provides a step-by-step guide to building a FastAPI application with CRUD operations.

## What you'll learn:
1. Install FastAPI and Uvicorn
2. Create a basic FastAPI app with a root endpoint
3. Add a POST endpoint to create items
4. Add a GET endpoint to list items with optional limit
5. Add a GET endpoint to retrieve specific items by ID with error handling
6. Test all endpoints using curl commands

## Prerequisites:
- Python installed on your system
- Basic understanding of REST APIs
- Terminal/Command Prompt access

## How to use this notebook:
1. Run the Python cells to understand the code structure
2. Copy and run the terminal commands (bat cells) to test the API
3. Make sure to start the server with `uvicorn main:app --reload` before testing endpoints

In [1]:
%pip install fastapi
%pip install uvicorn

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeableNote: you may need to restart the kernel to use updated packages.



In [ ]:
# Step 1: Basic FastAPI Setup with imports and Pydantic model
from fastapi import FastAPI, HTTPException, Query
from pydantic import BaseModel

app = FastAPI()

# Define the Item model using Pydantic
class Item(BaseModel):
    text: str
    is_done: bool = False

# Initialize empty list to store items
items = []

@app.get("/")
def root():
    return {"Hello": "World"}

In [ ]:
uvicorn main:app --reload

In [ ]:
curl -X POST -H "Content-Type: application/json" 'http://127.0.0.1:8000/items?item=apple'

In [ ]:
# Step 2: Add POST endpoint to create items using Pydantic model
@app.post("/items")
def create_item(item: Item):
    items.append(item)
    return items

In [ ]:
# Test the POST /items endpoint with JSON data
curl -X POST "http://127.0.0.1:8000/items" -H "Content-Type: application/json" -d "{\"text\": \"Buy groceries\", \"is_done\": false}"

In [ ]:
# Step 3: Add GET endpoint to list all items with response model
@app.get("/items", response_model=list[Item])
def list_items(limit: int = 10):
    return items[:limit]

In [ ]:
# Test the GET /items endpoint (list all items)
curl "http://127.0.0.1:8000/items"

# Test with limit parameter
curl "http://127.0.0.1:8000/items?limit=5"

In [ ]:
# Step 4: Add GET endpoint to retrieve a specific item by ID with response model
@app.get("/items/{item_id}", response_model=Item)
def get_item(item_id: int) -> Item:
    if item_id < 0 or item_id >= len(items):
        raise HTTPException(status_code=404, detail="Item not found")
    else:
        item = items[item_id]
        return item

In [ ]:
# Test the GET /items/{item_id} endpoint
# First add some items using JSON data, then try to get them

# Add a few items with JSON data
curl -X POST "http://127.0.0.1:8000/items" -H "Content-Type: application/json" -d "{\"text\": \"Buy groceries\", \"is_done\": false}"
curl -X POST "http://127.0.0.1:8000/items" -H "Content-Type: application/json" -d "{\"text\": \"Walk the dog\", \"is_done\": true}"
curl -X POST "http://127.0.0.1:8000/items" -H "Content-Type: application/json" -d "{\"text\": \"Read a book\", \"is_done\": false}"

# Get item by valid ID (0, 1, 2)
curl "http://127.0.0.1:8000/items/0"
curl "http://127.0.0.1:8000/items/1"

# Test error handling with invalid ID
curl "http://127.0.0.1:8000/items/99"

In [ ]:
# Complete FastAPI Application - Final Version
# This matches the structure in main.py

from fastapi import FastAPI, HTTPException, Query
from pydantic import BaseModel

app = FastAPI()

class Item(BaseModel):
    text: str
    is_done: bool = False

items = []

@app.get("/")
def root():
    return {"Hello": "World"}

@app.post("/items")
def create_item(item: Item):
    items.append(item)
    return items

@app.get("/items", response_model=list[Item])
def list_items(limit: int = 10):
    return items[:limit]

@app.get("/items/{item_id}", response_model=Item)
def get_item(item_id: int) -> Item:
    if item_id < 0 or item_id >= len(items):
        raise HTTPException(status_code=404, detail="Item not found")
    else:
        item = items[item_id]
        return item

In [ ]:
# Complete Testing Suite - Run these commands in terminal

# 1. Start the server (run this in a separate terminal)
# uvicorn main:app --reload

# 2. Test all endpoints in order:

# Test root endpoint
curl "http://127.0.0.1:8000/"

# Add some items using JSON data (matching the Pydantic model)
curl -X POST "http://127.0.0.1:8000/items" -H "Content-Type: application/json" -d "{\"text\": \"Buy groceries\", \"is_done\": false}"
curl -X POST "http://127.0.0.1:8000/items" -H "Content-Type: application/json" -d "{\"text\": \"Walk the dog\", \"is_done\": true}"
curl -X POST "http://127.0.0.1:8000/items" -H "Content-Type: application/json" -d "{\"text\": \"Read a book\", \"is_done\": false}"
curl -X POST "http://127.0.0.1:8000/items" -H "Content-Type: application/json" -d "{\"text\": \"Write code\", \"is_done\": true}"

# List all items
curl "http://127.0.0.1:8000/items"

# List items with limit
curl "http://127.0.0.1:8000/items?limit=2"

# Get specific items by ID
curl "http://127.0.0.1:8000/items/0"
curl "http://127.0.0.1:8000/items/2"

# Test error handling
curl "http://127.0.0.1:8000/items/99"
curl "http://127.0.0.1:8000/items/-1"